In [3]:
from langgraph.graph import StateGraph, START, END
from langchain_google_genai import ChatGoogleGenerativeAI
from dotenv import load_dotenv
from typing import TypedDict

In [5]:
# define model

load_dotenv()

model = ChatGoogleGenerativeAI(model= "gemini-2.5-flash")



In [11]:
# define state

class BlogState(TypedDict):
    topic: str
    outline: str
    score: float
    blog: str

In [ ]:
def get_outline(state: BlogState) -> BlogState:
    topic = state['topic']
    prompt = f"Generate outline for the Topic: {topic}"

    result = model.invoke(prompt).content

    state['outline'] = result
    return state

In [ ]:
def get_blog(state: BlogState) -> BlogState:
    topic = state['topic']
    outline = state['outline']
    prompt = f"Generate blog for the following Topic: {topic} & outline: \n {outline}"

    result = model.invoke(prompt).content

    state['blog'] = result
    return state

In [ ]:
def get_score(state: BlogState) -> BlogState:
    topic = state['topic']
    outline = state['outline']
    blog = state['blog']
    prompt = f"Give the score for the following blog on topic: {topic} and outline: \n {outline}. \n blog: {blog}"

    result = model.invoke(prompt).content

    state['score'] = result
    return state

In [18]:
# define graph
graph = StateGraph(BlogState)

# define node
graph.add_node("create_outline", get_outline)
graph.add_node("create_blog", get_blog)
graph.add_node("score", get_score)

#define edge
graph.add_edge(START, 'create_outline')
graph.add_edge('create_outline', 'create_blog')
graph.add_edge('create_blog', 'score')
graph.add_edge('score', END)

workflow = graph.compile()


In [20]:
intial_state = {'topic': 'Rise of AI in India'}

final_state = workflow.invoke(intial_state)

print(final_state)

{'topic': 'Rise of AI in India', 'outline': AIMessage(content='Here\'s a comprehensive outline for the topic "Rise of AI in India," covering its various facets from drivers to challenges and future outlook.\n\n---\n\n**Outline: The Rise of Artificial Intelligence in India**\n\n**I. Introduction**\n    A. Global Context: The AI Revolution and its significance.\n    B. India\'s Unique Position: A large, young, digitally-savvy population with significant data generation.\n    C. Definition of AI (in the Indian context): Focus on practical applications and transformative potential.\n    D. Thesis Statement: India is witnessing a rapid ascent in AI adoption and innovation, driven by diverse factors, but faces critical challenges requiring strategic interventions for inclusive and responsible growth.\n    E. Outline Overview: Briefly introduce the key areas to be covered.\n\n**II. Historical Context and Evolution of AI in India**\n    A. Early Stages: Academic interest, limited practical app

In [22]:
print(final_state['blog'].content)

## India's Digital Leap: The Unstoppable Rise of Artificial Intelligence

The world is in the throes of an Artificial Intelligence revolution, a transformative tide reshaping industries, economies, and societies. While global giants lead the charge, a new, formidable player is rapidly emerging on the AI landscape: India. With its massive, young, and digitally-savvy population, coupled with an unparalleled data generation capacity, India isn't just adopting AI; it's actively shaping its future.

This isn't merely about technology; it's about leveraging AI for practical applications and unlocking its immense transformative potential across every facet of Indian life. India is witnessing a rapid ascent in AI adoption and innovation, driven by diverse factors, but faces critical challenges requiring strategic interventions for inclusive and responsible growth. Let's delve into how the nation is embracing this technological wave, from its foundational digital push to its future as a global 